**Workshop notebooks:** [01 — Matching & Loading](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/01_matching_and_loading.ipynb)  &nbsp;·&nbsp; [02 — Network Overlay & Enrichment](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/02_network_overlay_enrichment.ipynb) &nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb) &nbsp;·&nbsp; **04 — Disease Modules (optional) (you are here)**

# 04 — Disease Modules & Network Overlap *(optional)*

**Network Medicine Workshop · Kidney Disease · Optional Part 4**

This notebook is an extension. It has two parts:

**Part A — Is your module really a module?** Is your DE nodes' connectivity in each network more than you'd expect from a random set of the same size and degree distribution?

**Part B — How does your disease relate to other diseases, topologically?** This is the classic network medicine "disease module overlap" idea (Menche et al., *Science* 2015): two diseases whose gene modules sit close together in the interactome tend to share mechanisms, comorbidities, and sometimes treatments — even if the gene lists themselves barely overlap. We compare our C3 glomerulopathy module against:
- **Atypical hemolytic uremic syndrome (aHUS)** — a *different* kidney disease, but one driven by the same underlying mechanism (dysregulation of the alternative complement pathway), expected to show network *proximity*
- **An unrelated disease** (default: Parkinson's disease, change if you'd like) — expected to show network *separation*

The statistic is the **network separation S_AB**:

```
S_AB = <d_AB> - (<d_AA> + <d_BB>) / 2
```

`d_AA` / `d_BB` = average shortest-path distance between all two nodes in the same module. `d_AB` = average shortest-path distance between all two nodes in the two different modules. **S_AB < 0 means the modules overlap/sit close together** (shared mechanism); **S_AB > 0 means they're topologically separated** (distinct mechanisms).

Requires Notebooks 1–2 (and ideally 3, for the richest version of "your module").


**New here?** Run cells top to bottom with Shift+Enter (or the ▶ button), and wait for each one to finish before running the next — later cells depend on variables set by earlier ones. See Notebook 1 for a fuller intro to Colab if this is your first time.

## Setup

Same pattern as the earlier notebooks: install this notebook's packages, reconnect to Drive, and reload what previous notebooks computed — here, the per-layer modules from Notebook 2, plus (ideally) the bridge module from Notebook 3.

In [ ]:
!pip install -q scipy networkx requests mygene matplotlib


Reconnect to Drive, exactly as before — click through the permission prompt if it appears.

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')


With Drive connected, load back the per-layer modules (seed nodes + connected nodes) that Notebook 2 saved, along with the three network graphs.

In [ ]:
import os, sys, pickle, time
import pandas as pd
import numpy as np
import networkx as nx
import requests

BASE_DIR = "/content/drive/MyDrive/network_medicine_workshop"
BASE_DIR = '.'
PROC_DIR = os.path.join(BASE_DIR, "processed")
LOOKUPS_DIR = os.path.join(BASE_DIR, "lookups")

sys.path.insert(0, os.path.join(BASE_DIR, "helpers"))
_helper_file = os.path.join(BASE_DIR, "helpers", "nb4_helpers.py")
if not os.path.exists(_helper_file):
    # Rare, but can happen if Drive's sync lags behind a fresh download, or this notebook runs
    # before Notebook 1 has -- fetch this one helper file directly instead of leaving you stuck.
    import requests, importlib
    r = requests.get(f"https://raw.githubusercontent.com/marlene-grabner/NetworkMedicine_Workshop/main/helpers/nb4_helpers.py", timeout=30)
    r.raise_for_status()
    with open(_helper_file, "wb") as f:
        f.write(r.content)
    importlib.invalidate_caches()
    print(f"[recovered] fetched missing nb4_helpers.py directly")

from nb4_helpers import (degree_bins, plot_connectivity_test, plot_s_ab_comparison,
                         fetch_disease_genes, symbols_to_ncbi_in_network)  # see helpers/nb4_helpers.py

with open(os.path.join(PROC_DIR, "modules.pkl"), "rb") as f:
    modules = pickle.load(f)

graphs = {}
for layer in ["ppi", "transcriptome", "metabolite"]:
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "rb") as f:
        graphs[layer] = pickle.load(f)

print("Loaded modules for layers:", list(modules.keys()))


---
# Part A — Module significance

For each network layer, we test whether the DE-mapped nodes are more connected (bigger largest connected component) than a random node set drawn from the same **degree distribution** — controlling for the fact that hub nodes are just generally more "connected" regardless of biology.


The permutation test itself is the two functions below. `degree_bins` (imported from `helpers/nb4_helpers.py` — it's a plumbing detail of how we draw degree-matched random node sets, not the statistic itself) groups network nodes into buckets of similar degree. `largest_cc_size` is a small helper we already used in Notebook 2. `module_significance` ties them together: it repeatedly draws degree-matched random node sets (1000 times, by default), records how large their largest connected component is each time, and compares that "null distribution" to your observed value.

In [ ]:
def largest_cc_size(G, nodes):
    if len(nodes) < 2:
        return len(nodes)
    sub = G.subgraph(nodes)
    if sub.number_of_edges() == 0:
        return 1
    return len(max(nx.connected_components(sub), key=len))

def module_significance(G, seed_nodes, n_perm=1000, seed=0):
    rng = np.random.default_rng(seed)
    bin_of_node, nodes_in_bin = degree_bins(G)
    observed = largest_cc_size(G, seed_nodes)

    bin_counts = {}
    for n in seed_nodes:
        b = bin_of_node[n]
        bin_counts[b] = bin_counts.get(b, 0) + 1

    null_dist = []
    for _ in range(n_perm):
        random_set = []
        for b, count in bin_counts.items():
            pool = nodes_in_bin[b]
            random_set.extend(rng.choice(pool, size=min(count, len(pool)), replace=False))
        null_dist.append(largest_cc_size(G, random_set))

    null_dist = np.array(null_dist)
    z = (observed - null_dist.mean()) / (null_dist.std() + 1e-9)
    p = (null_dist >= observed).mean()
    return {"observed_lcc": observed, "null_mean": null_dist.mean(),
            "null_std": null_dist.std(), "z_score": z, "p_value": p, "null_dist": null_dist}


Now we run that test once per network layer (skipping any layer with fewer than 5 mapped nodes, since the statistics get unreliable below that) and print the observed-vs-random comparison for each.

In [ ]:
connectivity_results = {}
for layer, G in graphs.items():
    seed_nodes = modules[layer]["seed_nodes"]
    if len(seed_nodes) < 5:
        print(f"[{layer}] too few mapped nodes ({len(seed_nodes)}) to test — skipping")
        continue
    res = module_significance(G, seed_nodes, n_perm=5000)
    connectivity_results[layer] = res
    print(f"[{layer}] observed LCC={res['observed_lcc']} vs random {res['null_mean']:.1f} ± {res['null_std']:.1f}  "
          f"→ z={res['z_score']:.2f}, p={res['p_value']:.4f}")


Let's visualize this: each panel below is one network layer's null distribution (1000 random degree-matched draws), with your observed value marked in red. The further right the red line sits from the gray histogram, the more surprising your module's connectivity is.

In [ ]:
plot_connectivity_test(connectivity_results, PROC_DIR)


A large positive z-score (small p-value) says these genes/proteins/metabolites cluster together in the network far more than chance. Thid id evidence of a real, shared underlying process. A z-score near zero says the DE list, at least as mapped onto this network, doesn't behave like a coherent module.


---
# Part B — Disease-disease network separation (S_AB)

## Step 1 — Choose "your" module

We use the richest available representation of your kidney-disease signal: the **cross-omics bridge module** from Notebook 3 if you've run it (protein module + metabolite-linked genes + connecting nodes), falling back to the plain PPI protein module otherwise.


In [ ]:
bridge_path = os.path.join(PROC_DIR, "bridge_module.pkl")
if os.path.exists(bridge_path):
    with open(bridge_path, "rb") as f:
        MODULE_A = pickle.load(f)
    module_a_source = "bridge module (Notebook 3)"
else:
    MODULE_A = modules["ppi"]["seed_nodes"]
    module_a_source = "PPI protein module (Notebook 2)"

G = graphs["ppi"]
MODULE_A = MODULE_A & set(G.nodes())
print(f"Module A = {module_a_source}: {len(MODULE_A)} nodes in the PPI network")


## Step 2 — Fetch reference disease gene sets from Open Targets

Same approach as Notebook 3: search for a disease name, get its EFO ID, pull its associated target genes. Both reference diseases below are precomputed and bundled in `lookups/`, so this doesn't call Open Targets by default — point `DISEASE_SIMILAR`/`DISEASE_DIFFERENT` at something else and it falls back to a live query automatically for whichever one isn't cached.


In [ ]:
DISEASE_SIMILAR = "Atypical hemolytic uremic syndrome"   # a different disease, same complement pathway -> expected network-close
DISEASE_DIFFERENT = "Parkinson disease"   # expected to be network-far

# fetch_disease_genes() checks lookups/opentargets_diseases.json first, and only falls back to
# a live Open Targets search + association lookup if a disease isn't cached there — see helpers/nb4_helpers.py
_, similar_gene_symbols = fetch_disease_genes(DISEASE_SIMILAR, lookups_dir=LOOKUPS_DIR)
_, different_gene_symbols = fetch_disease_genes(DISEASE_DIFFERENT, lookups_dir=LOOKUPS_DIR)


## Step 3 — Map disease genes to NCBI Gene IDs and restrict to the PPI network

Same batch-mapping approach as Notebook 1 — precomputed lookup first, live mygene.info call only for whatever that doesn't cover.


In [ ]:
# symbols_to_ncbi_in_network() is the same batch gene-symbol matching from Notebook 1,
# just wrapped up — see helpers/nb4_helpers.py
MODULE_B = symbols_to_ncbi_in_network(similar_gene_symbols, G, lookups_dir=LOOKUPS_DIR)     # aHUS
MODULE_C = symbols_to_ncbi_in_network(different_gene_symbols, G, lookups_dir=LOOKUPS_DIR)   # unrelated disease

print(f"Module B ({DISEASE_SIMILAR}): {len(MODULE_B)} nodes in PPI network")
print(f"Module C ({DISEASE_DIFFERENT}): {len(MODULE_C)} nodes in PPI network")
print(f"Direct gene overlap A∩B: {len(MODULE_A & MODULE_B)}   A∩C: {len(MODULE_A & MODULE_C)}")


## Step 4 — Compute network separation S_AB

We need, for two modules A and B: the average within-module distance for each (`d_AA`, `d_BB`) and the average between-module distance (`d_AB`). All of this comes from two shortest-path searches (one starting from every gene in A, one from every gene in B) rather than computing every possible pairwise distance directly, which would be far slower.


The setup mirrors Notebook 3's bridging step: convert the network into an efficient numerical form once, then reuse the same fast shortest-path search for every distance we need (within-module and between-module) instead of computing every pairwise distance directly, which would be far more expensive at this network size. `network_separation` below ties this into the `S_AB` formula from the introduction above.

In [ ]:
from scipy.sparse.csgraph import dijkstra

nodes = list(G.nodes())
node_idx = {n: i for i, n in enumerate(nodes)}
Adj = nx.to_scipy_sparse_array(G, nodelist=nodes, weight=None, format="csr")

def multi_source_distances(module):
    idx = [node_idx[n] for n in module if n in node_idx]
    return dijkstra(csgraph=Adj, directed=False, indices=idx, unweighted=True), idx

def network_separation(G, module_a, module_b, label_a="A", label_b="B"):
    dist_from_a, idx_a = multi_source_distances(module_a)   # shape (|A|, n_nodes)
    dist_from_b, idx_b = multi_source_distances(module_b)

    # d_AA: for each node in A, min distance to another node in A (exclude self), averaged
    sub_aa = dist_from_a[:, idx_a].copy()
    np.fill_diagonal(sub_aa, np.inf)
    d_aa = np.nanmean(np.min(sub_aa, axis=1)[np.isfinite(np.min(sub_aa, axis=1))])

    sub_bb = dist_from_b[:, idx_b].copy()
    np.fill_diagonal(sub_bb, np.inf)
    d_bb = np.nanmean(np.min(sub_bb, axis=1)[np.isfinite(np.min(sub_bb, axis=1))])

    # d_AB: distance from each A node to nearest B node, AND each B node to nearest A node, pooled
    sub_ab = dist_from_a[:, idx_b]   # rows=A, cols=B
    d_a_to_b = np.min(sub_ab, axis=1)
    d_b_to_a = np.min(sub_ab, axis=0)   # symmetric graph, so this equals "each B node's min distance to A"
    pooled = np.concatenate([d_a_to_b, d_b_to_a])
    d_ab = np.nanmean(pooled[np.isfinite(pooled)])

    s_ab = d_ab - (d_aa + d_bb) / 2
    print(f"[{label_a} vs {label_b}]  d_AA={d_aa:.2f}  d_BB={d_bb:.2f}  d_AB={d_ab:.2f}  ->  S_AB = {s_ab:.3f}")
    return {"d_AA": d_aa, "d_BB": d_bb, "d_AB": d_ab, "S_AB": s_ab}


Now we compute it twice: your module against the related disease (aHUS — a different disease, same complement pathway), and your module against the unrelated one (Parkinson's, by default) — the whole point of Part B is comparing these two numbers.

In [ ]:
sep_similar = network_separation(G, MODULE_A, MODULE_B, "your module", DISEASE_SIMILAR)
sep_different = network_separation(G, MODULE_A, MODULE_C, "your module", DISEASE_DIFFERENT)


**Interpretation:** `S_AB < 0` → the two modules overlap / sit close together in the network → shared mechanism. `S_AB > 0` → the modules are topologically separated → distinct mechanisms. If the biology behaves the way we'd expect, `S_AB(your module, aHUS)` should be noticeably smaller (more negative / less positive) than `S_AB(your module, {unrelated disease})`, since C3G and aHUS both stem from dysregulated alternative-pathway complement activity even though they are clinically distinct diseases. Our C3G module should sit closer, in network space, to another complement-mediated kidney disease than to an unrelated one.

This is also a nice moment to contrast with the **direct gene overlap** numbers from Step 3: it's common for two related diseases to show striking network proximity (small/negative S_AB) even when they share almost no genes directly. That's the whole point of doing this in network space rather than just intersecting lists.


One last plot to make the comparison visible at a glance: a negative (green) bar means that disease's module sits close to yours in the network; a positive (red) bar means it sits apart.

In [ ]:
labels = [DISEASE_SIMILAR, DISEASE_DIFFERENT]
values = [sep_similar["S_AB"], sep_different["S_AB"]]
plot_s_ab_comparison(labels, values, PROC_DIR)


---


**Workshop notebooks:** [01 — Matching & Loading](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/01_matching_and_loading.ipynb)  &nbsp;·&nbsp; [02 — Network Overlay & Enrichment](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/02_network_overlay_enrichment.ipynb) &nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb) &nbsp;·&nbsp; **04 — Disease Modules (optional) (you are here)**